In [ ]:
# âœ… STEP 1: Upload kaggle.json
from google.colab import files
files.upload()  # Upload your kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# âœ… STEP 3: Clean install YOLOv5 and dependencies

# Ensure you're in a good directory
%cd /content

# Remove previous yolov5 folder if corrupted
!rm -rf yolov5

# Clone the YOLOv5 repository fresh
!git clone --depth 1 https://github.com/ultralytics/yolov5

# Change to the yolov5 directory
%cd yolov5

# Install all required dependencies cleanly
!pip install -r requirements.txt


In [ ]:
# âœ… STEP 4: Download & unzip datasets
!kaggle datasets download -d ankanghosh651/object-detection-wildlife-dataset-yolo-format
!unzip -q object-detection-wildlife-dataset-yolo-format.zip -d /content/wildlife

!kaggle datasets download -d trainingdatapro/miners-detection
!unzip -q miners-detection.zip -d /content/miners

In [ ]:
# âœ… STEP 5: Combine datasets
import os, shutil
from glob import glob
import xml.etree.ElementTree as ET

!mkdir -p /content/combined_dataset/images/train
!mkdir -p /content/combined_dataset/labels/train

def copy_imgs(src, dst):
    for path in glob(f"{src}/*"):
        if path.endswith(('.jpg', '.png')):
            shutil.copy(path, dst)

def remap_labels(src, dst, offset=0):
    for label_path in glob(f"{src}/*.txt"):
        with open(label_path, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0]) + offset
                new_lines.append(f"{cls_id} {' '.join(parts[1:])}\n")
        with open(os.path.join(dst, os.path.basename(label_path)), 'a') as f:
            f.writelines(new_lines)

wildlife_imgs = glob("/content/wildlife/**/images", recursive=True)[0]
wildlife_lbls = glob("/content/wildlife/**/labels", recursive=True)[0]
copy_imgs(wildlife_imgs, "/content/combined_dataset/images/train")
remap_labels(wildlife_lbls, "/content/combined_dataset/labels/train")

miners_img_dir = "/content/miners/boxes"
miners_lbl_dir = "/content/miners/labels"
os.makedirs(miners_lbl_dir, exist_ok=True)

def convert_xml_to_yolo(xml_file, img_folder, label_output, class_id):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    size = root.find("size")
    if size is None: return
    width = int(size.find("width").text)
    height = int(size.find("height").text)
    label_path = os.path.join(label_output, root.find("filename").text.rsplit(".", 1)[0] + ".txt")
    with open(label_path, "w") as f:
        for obj in root.findall("object"):
            bbox = obj.find("bndbox")
            xmin = int(bbox.find("xmin").text)
            xmax = int(bbox.find("xmax").text)
            ymin = int(bbox.find("ymin").text)
            ymax = int(bbox.find("ymax").text)
            x_c = ((xmin + xmax) / 2) / width
            y_c = ((ymin + ymax) / 2) / height
            w = (xmax - xmin) / width
            h = (ymax - ymin) / height
            f.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

for ann in glob("/content/miners/*.xml"):
    convert_xml_to_yolo(ann, miners_img_dir, miners_lbl_dir, class_id=4)

copy_imgs(miners_img_dir, "/content/combined_dataset/images/train")
remap_labels(miners_lbl_dir, "/content/combined_dataset/labels/train", offset=0)


In [ ]:
# âœ… STEP 6: Create data.yaml
yaml_text = """
path: /content/combined_dataset
train: images/train
val: images/train

names:
  0: buffalo
  1: elephant
  2: rhino
  3: zebra
  4: miner
"""
with open("/content/yolov5/data/data.yaml", "w") as f:
    f.write(yaml_text)

In [ ]:
# âœ… Patch PyTorch 2.6+ to allow YOLOv5 model unpickling
from models.yolo import Model
import torch.serialization
torch.serialization.add_safe_globals({'Model': Model})


In [ ]:
!python train.py --img 640 --batch 16 --epochs 80 --data data/data.yaml --weights yolov5m.pt --name tuned_model --hyp data/hyps/hyp.scratch-low.yaml


In [ ]:
# STEP 8ï¸âƒ£: Display results
import glob
from IPython.display import Image, display

output_dirs = sorted(glob.glob('/content/yolov5/runs/train/tuned_model*'), key=os.path.getmtime)
if output_dirs:
    last_run = output_dirs[-1]
    print(f"âœ… Training logs in: {last_run}")
    result_imgs = glob.glob(f"{last_run}/*.jpg") + glob.glob(f"{last_run}/*.png")
    if result_imgs:
        display(Image(result_imgs[0]))
    else:
        print("âš ï¸ No result images found.")
else:
    print("âŒ No output directory found.")


In [ ]:
# ðŸ§  Check where YOLO actually saved the model
import glob

model_paths = glob.glob('/content/yolov5/runs/train/*/weights/best.pt')
print("ðŸ“¦ Found model weights at:")
for path in model_paths:
    print(path)


In [ ]:
# âœ… STEP 10: Download Trained Weights (auto-path)
from google.colab import files
import glob

model_paths = glob.glob('/content/yolov5/runs/train/*/weights/best.pt')
if model_paths:
    latest_model = sorted(model_paths, key=os.path.getmtime)[-1]
    print(f"ðŸ“¦ Found model weights at: {latest_model}")
    files.download(latest_model)
else:
    print("âŒ No model weights found! Please ensure training completed without error.")

